In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/JobShield_LK"
)

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
EXPERIMENTS_DIR = PROJECT_DIR / "experiments"

for folder in [
    RAW_DIR,
    PROCESSED_DIR,
    MODELS_DIR,
    RESULTS_DIR,
    EXPERIMENTS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Folders created successfully.")

Project directory: /content/drive/MyDrive/JobShield_LK
Folders created successfully.


In [3]:
%pip -q install \
    pandas \
    numpy \
    scikit-learn \
    matplotlib \
    seaborn \
    joblib \
    huggingface_hub \
    sentence-transformers

SyntaxError: invalid syntax (3805981368.py, line 68)

In [ ]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nFraudulent column:")
print(df["fraudulent"].value_counts())

print("\nClass percentages:")
print(
    df["fraudulent"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

SyntaxError: invalid syntax (3805981368.py, line 68)

In [6]:
raw_path = RAW_DIR / "emscad_original.csv"

df.to_csv(raw_path, index=False)

print("Saved to:", raw_path)

Saved to: /content/drive/MyDrive/JobShield_LK/data/raw/emscad_original.csv


In [7]:
TEXT_COLUMNS = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

required_columns = TEXT_COLUMNS + ["fraudulent"]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

data = df[required_columns].copy()

data["text"] = (
    data[TEXT_COLUMNS]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

data["label"] = data["fraudulent"].astype(int)

data = data[
    (data["text"].str.len() > 0)
    & data["label"].isin([0, 1])
].copy()

data = data[["text", "label"]].reset_index(drop=True)

print("Prepared shape:", data.shape)

print("\nClass distribution:")
print(data["label"].value_counts())

data.head()

Prepared shape: (17880, 2)

Class distribution:
label
0    17014
1      866
Name: count, dtype: int64


,text,label
0,"Marketing Intern We're Food52, and we've creat...",0
1,Customer Service - Cloud Video Production 90 S...,0
2,Commissioning Machinery Assistant (CMA) Valor ...,0
3,Account Executive - Washington DC Our passion ...,0
4,Bill Review Manager SpotSource Solutions LLC i...,0


In [8]:
from sklearn.model_selection import train_test_split

train_val_df, test_df = train_test_split(
    data,
    test_size=0.15,
    random_state=42,
    stratify=data["label"]
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.1764706,
    random_state=42,
    stratify=train_val_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Test records:", len(test_df))

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

Training records: 12515
Validation records: 2683
Test records: 2682

Training labels:
label
0    11909
1      606
Name: count, dtype: int64

Validation labels:
label
0    2553
1     130
Name: count, dtype: int64

Test labels:
label
0    2552
1     130
Name: count, dtype: int64


In [9]:
train_df.to_csv(
    PROCESSED_DIR / "emscad_train.csv",
    index=False
)

val_df.to_csv(
    PROCESSED_DIR / "emscad_validation.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_DIR / "emscad_test.csv",
    index=False
)

print("All partitions saved.")

All partitions saved.


In [11]:
import time
import json

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

experiment_results = []


def evaluate_model(
    model,
    X,
    y,
    experiment_name,
    threshold=0.5,
    training_time=None
):
    start = time.perf_counter()

    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X)[:, 1]

    elif hasattr(model, "decision_function"):
        # Decision scores are not probabilities.
        # Use them for ranking metrics, but don't interpret
        # them as calibrated probabilities.
        scores = model.decision_function(X)

    else:
        raise ValueError(
            "Model must provide predict_proba or decision_function."
        )

    prediction_time = time.perf_counter() - start

    # For decision_function models, threshold=0.0 is the
    # natural default, not 0.5.
    predictions = (
        scores >= threshold
    ).astype(int)

    result = {
        "experiment": experiment_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y, predictions),
        "fraud_precision": precision_score(
            y, predictions, pos_label=1, zero_division=0
        ),
        "fraud_recall": recall_score(
            y, predictions, pos_label=1, zero_division=0
        ),
        "fraud_f1": f1_score(
            y, predictions, pos_label=1, zero_division=0
        ),
        "macro_f1": f1_score(
            y, predictions, average="macro", zero_division=0
        ),
        "average_precision": average_precision_score(y, scores),
        "roc_auc": roc_auc_score(y, scores),
        "training_seconds": training_time,
        "prediction_seconds": prediction_time,
    }

    experiment_results.append(result)

    print(f"\n===== {experiment_name} =====")

    print(
        json.dumps(
            result,
            indent=2,
            default=float
        )
    )

    print("\nClassification report:")

    print(
        classification_report(
            y,
            predictions,
            labels=[0, 1],
            target_names=["Legitimate", "Fraudulent"],
            digits=4,
            zero_division=0
        )
    )

    print("Confusion matrix:")
    print(confusion_matrix(y, predictions, labels=[0, 1]))

    return result

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

e1_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_features=100_000,
            sublinear_tf=True,
            dtype=np.float32
        )
    ),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

start = time.perf_counter()

e1_model.fit(
    train_df["text"],
    train_df["label"]
)

e1_training_time = time.perf_counter() - start

print("E1 training time:", round(e1_training_time, 2), "seconds")

E1 training time: 12.82 seconds


In [13]:
e1_result = evaluate_model(
    model=e1_model,
    X=val_df["text"],
    y=val_df["label"],
    experiment_name="E1_Word_TFIDF_LogisticRegression",
    threshold=0.5,
    training_time=e1_training_time
)


===== E1_Word_TFIDF_LogisticRegression =====
{
  "experiment": "E1_Word_TFIDF_LogisticRegression",
  "threshold": 0.5,
  "accuracy": 0.9858367499068207,
  "fraud_precision": 0.8484848484848485,
  "fraud_recall": 0.8615384615384616,
  "fraud_f1": 0.8549618320610687,
  "macro_f1": 0.923758345497619,
  "average_precision": 0.8983250906926561,
  "roc_auc": 0.9793154358371751,
  "training_seconds": 12.824364074000187,
  "prediction_seconds": 1.7227793360000305
}

Classification report:
              precision    recall  f1-score   support

  Legitimate     0.9929    0.9922    0.9926      2553
  Fraudulent     0.8485    0.8615    0.8550       130

    accuracy                         0.9858      2683
   macro avg     0.9207    0.9269    0.9238      2683
weighted avg     0.9859    0.9858    0.9859      2683

Confusion matrix:
[[2533   20]
 [  18  112]]


In [14]:
from sklearn.pipeline import FeatureUnion

e2_features = FeatureUnion([
    (
        "word",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_features=100_000,
            sublinear_tf=True,
            dtype=np.float32
        )
    ),
    (
        "char",
        TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=2,
            max_features=100_000,
            sublinear_tf=True,
            dtype=np.float32
        )
    )
])

e2_model = Pipeline([
    ("features", e2_features),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

start = time.perf_counter()

e2_model.fit(
    train_df["text"],
    train_df["label"]
)

e2_training_time = time.perf_counter() - start

print("E2 training time:", round(e2_training_time, 2), "seconds")

E2 training time: 78.56 seconds


In [15]:
e2_result = evaluate_model(
    model=e2_model,
    X=val_df["text"],
    y=val_df["label"],
    experiment_name="E2_Word_Char_TFIDF_LogisticRegression",
    threshold=0.5,
    training_time=e2_training_time
)


===== E2_Word_Char_TFIDF_LogisticRegression =====
{
  "experiment": "E2_Word_Char_TFIDF_LogisticRegression",
  "threshold": 0.5,
  "accuracy": 0.9854640327991054,
  "fraud_precision": 0.8473282442748091,
  "fraud_recall": 0.8538461538461538,
  "fraud_f1": 0.8505747126436781,
  "macro_f1": 0.9214675717968636,
  "average_precision": 0.9070128588894337,
  "roc_auc": 0.9815179728223208,
  "training_seconds": 78.55933505200005,
  "prediction_seconds": 14.631901143999812
}

Classification report:
              precision    recall  f1-score   support

  Legitimate     0.9926    0.9922    0.9924      2553
  Fraudulent     0.8473    0.8538    0.8506       130

    accuracy                         0.9855      2683
   macro avg     0.9199    0.9230    0.9215      2683
weighted avg     0.9855    0.9855    0.9855      2683

Confusion matrix:
[[2533   20]
 [  19  111]]


In [16]:
from sklearn.svm import LinearSVC

e3_model = Pipeline([
    ("features", e2_features),
    (
        "classifier",
        LinearSVC(
            class_weight="balanced",
            random_state=42,
            max_iter=3000
        )
    )
])

start = time.perf_counter()

e3_model.fit(
    train_df["text"],
    train_df["label"]
)

e3_training_time = time.perf_counter() - start

print("E3 training time:", round(e3_training_time, 2), "seconds")

E3 training time: 94.36 seconds


In [17]:
e3_result = evaluate_model(
    model=e3_model,
    X=val_df["text"],
    y=val_df["label"],
    experiment_name="E3_Word_Char_TFIDF_LinearSVM",
    threshold=0.0,
    training_time=e3_training_time
)


===== E3_Word_Char_TFIDF_LinearSVM =====
{
  "experiment": "E3_Word_Char_TFIDF_LinearSVM",
  "threshold": 0.0,
  "accuracy": 0.9895639209839732,
  "fraud_precision": 0.9553571428571429,
  "fraud_recall": 0.823076923076923,
  "fraud_f1": 0.8842975206611571,
  "macro_f1": 0.9394165198934201,
  "average_precision": 0.9362028225281912,
  "roc_auc": 0.9830636656723614,
  "training_seconds": 94.35992559700003,
  "prediction_seconds": 14.594066637999958
}

Classification report:
              precision    recall  f1-score   support

  Legitimate     0.9911    0.9980    0.9945      2553
  Fraudulent     0.9554    0.8231    0.8843       130

    accuracy                         0.9896      2683
   macro avg     0.9732    0.9106    0.9394      2683
weighted avg     0.9893    0.9896    0.9892      2683

Confusion matrix:
[[2548    5]
 [  23  107]]


In [18]:
results_df = pd.DataFrame(experiment_results)

display(
    results_df[
        [
            "experiment",
            "accuracy",
            "fraud_precision",
            "fraud_recall",
            "fraud_f1",
            "macro_f1",
            "average_precision",
            "roc_auc",
            "training_seconds",
            "prediction_seconds"
        ]
    ].sort_values(
        by="fraud_f1",
        ascending=False
    )
)

,experiment,accuracy,fraud_precision,fraud_recall,fraud_f1,macro_f1,average_precision,roc_auc,training_seconds,prediction_seconds
2,E3_Word_Char_TFIDF_LinearSVM,0.989564,0.955357,0.823077,0.884298,0.939417,0.936203,0.983064,94.359926,14.594067
0,E1_Word_TFIDF_LogisticRegression,0.985837,0.848485,0.861538,0.854962,0.923758,0.898325,0.979315,12.824364,1.722779
1,E2_Word_Char_TFIDF_LogisticRegression,0.985464,0.847328,0.853846,0.850575,0.921468,0.907013,0.981518,78.559335,14.631901


In [19]:
results_df.to_csv(
    RESULTS_DIR / "experiment_results.csv",
    index=False
)

print("Results saved.")

Results saved.


In [20]:
import torch

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-small",
    device=device
)

print("Embedding model loaded.")

Device: cuda


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Embedding model loaded.


In [22]:
def create_embeddings(texts, batch_size=64):
    formatted_texts = [
        "passage: " + str(text)
        for text in texts
    ]

    embeddings = embedding_model.encode(
        formatted_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return embeddings.astype(np.float32)


X_train_e5 = create_embeddings(
    train_df["text"].tolist()
)

X_val_e5 = create_embeddings(
    val_df["text"].tolist()
)

print("Training embeddings:", X_train_e5.shape)
print("Validation embeddings:", X_val_e5.shape)

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Training embeddings: (12515, 384)
Validation embeddings: (2683, 384)


In [23]:
def create_embeddings(texts, batch_size=64):
    formatted_texts = [
        "passage: " + str(text)
        for text in texts
    ]

    embeddings = embedding_model.encode(
        formatted_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return embeddings.astype(np.float32)


X_train_e5 = create_embeddings(
    train_df["text"].tolist()
)

X_val_e5 = create_embeddings(
    val_df["text"].tolist()
)

print("Training embeddings:", X_train_e5.shape)
print("Validation embeddings:", X_val_e5.shape)

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Training embeddings: (12515, 384)
Validation embeddings: (2683, 384)


In [25]:
e4_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

start = time.perf_counter()

e4_model.fit(
    X_train_e5,
    train_df["label"]
)

e4_training_time = time.perf_counter() - start

print("E4 classifier training time:",
      round(e4_training_time, 2), "seconds")

E4 classifier training time: 0.29 seconds


In [26]:
e4_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

start = time.perf_counter()

e4_model.fit(
    X_train_e5,
    train_df["label"]
)

e4_training_time = time.perf_counter() - start

print("E4 classifier training time:",
      round(e4_training_time, 2), "seconds")

E4 classifier training time: 0.27 seconds


In [27]:
e4_result = evaluate_model(
    model=e4_model,
    X=X_val_e5,
    y=val_df["label"],
    experiment_name="E4_MultilingualE5_LogisticRegression",
    threshold=0.5,
    training_time=e4_training_time
)


===== E4_MultilingualE5_LogisticRegression =====
{
  "experiment": "E4_MultilingualE5_LogisticRegression",
  "threshold": 0.5,
  "accuracy": 0.8773760715616847,
  "fraud_precision": 0.26139088729016785,
  "fraud_recall": 0.8384615384615385,
  "fraud_f1": 0.39853747714808047,
  "macro_f1": 0.665133025770554,
  "average_precision": 0.57077162549568,
  "roc_auc": 0.9320196450631234,
  "training_seconds": 0.26929960600000413,
  "prediction_seconds": 0.0036954189999960363
}

Classification report:
              precision    recall  f1-score   support

  Legitimate     0.9907    0.8794    0.9317      2553
  Fraudulent     0.2614    0.8385    0.3985       130

    accuracy                         0.8774      2683
   macro avg     0.6261    0.8589    0.6651      2683
weighted avg     0.9554    0.8774    0.9059      2683

Confusion matrix:
[[2245  308]
 [  21  109]]
